<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Numbers, confirmation and recovery

**[Numbers, confirmation and recovery](https://learning.nextia-ai.com/courses/speech/m07/numbers-confirmation-and-recovery/)** · Speech AI: Transcribe, Understand and Speak · Module 7, lesson 5 of 6 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn a caller's spoken order number into digits, check it three ways before the IVR uses it, read it back in groups, and understand a yes or a no, safely.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download. |
| **You should know** | The order-number checks, from [Help the model with your words](https://learning.nextia-ai.com/courses/speech/m04/help-the-model-with-your-words/), and the IVR's speech path, from [Let callers say what they need](https://learning.nextia-ai.com/courses/speech/m07/let-callers-say-what-they-need/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m07/numbers-confirmation-and-recovery/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M07-L05-numbers-confirmation-and-recovery/numbers-confirmation-and-recovery-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 1 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/ivr/calls.jsonl": "5ee1fbad92c79b73fa908076325f2ce6d4030ed12439dacc26b6dcd9bc00d828",
    "data/recordings/agent/inputs.jsonl": "4a9274d0f9dab006ca40634525c761c27afc75d9f5b9f889501d5debe0c4c5d1",
    "data/orders.json": "74f313e9e6c6eac155c655f2c4ed5016ae8aa6076c38ca88edf7c09ed901aeda",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The order numbers the callers said

Run the next cell. It collects every caller turn of the IVR test calls that says an order number, with the words the caller said and what **Whisper base** wrote (with no state prompt). We count per caller turn: these calls hold only the caller's side, so nothing else can hide a wrong digit.

In [ ]:
ivr_calls = [json.loads(line) for line in open("data/ivr/calls.jsonl", encoding="utf-8") if line.strip()]
recognised = {}
for line in open("data/recordings/agent/inputs.jsonl", encoding="utf-8"):
    row = json.loads(line)
    recognised[row["call_id"]] = row
orders = {o["order"]: o for o in json.load(open("data/orders.json"))}

# Every caller turn that says an order number: counted per caller turn (these calls hold only the caller's side).
number_turns = []
for call in ivr_calls:
    for i, turn in enumerate(call["inputs"]):
        if turn.get("spoken"):
            number_turns.append({"call": call["call_id"], "language": call["language"], "number": turn["say"],
                                 "spoken": turn["spoken"], "heard": recognised[call["call_id"]]["turns"][i]["text"]})
print(len(number_turns), "caller turns with an order number;", len(orders), "orders in the order system")
for t in number_turns[:4]:
    print(f'{t["call"]} {t["language"]}  said {t["spoken"]!r:40} Whisper base wrote {t["heard"]!r}')

> **Check.** 31 caller turns with an order number, and 160 orders in Larkfield's test order system.

## 2. Words to digits, then the checks

`find_order_number` uses the normalisation guide of the course (`cp.normalise`) to turn spoken numbers into digits, keeps the longest run of digits, and checks its length and its check digit.

> **Predict.** "three oh three three six two": does "oh" become 0?

In [ ]:
import re

def luhn_ok(number):
    """The last digit is a check digit (Luhn): most one-digit mistakes fail."""
    if not number.isdigit() or len(number) < 2:
        return False
    total = 0
    for i, ch in enumerate(reversed(number)):
        d = int(ch) * (2 if i % 2 else 1)
        total += d - 9 if d > 9 else d
    return total % 10 == 0

def find_order_number(text, language="en", length=6):
    """(number, problem): the longest run of digits after the normalisation guide, and the first check it fails."""
    runs = re.findall(r"\d+", cp.normalise(text, "full", language))
    if not runs:
        return None, "no digits"
    best = max(runs, key=len)
    if len(best) != length:
        return best, "too short" if len(best) < length else "too long"
    if not luhn_ok(best):
        return best, "check digit wrong"
    return best, None

for text in ["seven nine eight zero nine zero", "three oh three three six two", "Two two nine one two.",
             "114940."]:
    print(repr(text), find_order_number(text))

> **Check.** "seven nine eight zero nine zero" and "three oh three three six two" give valid numbers. "Two two nine one two." is too short. "114940." fails the check digit (the caller said 114942).

## 3. How many numbers are right, and what stopped the others

The third check is the **order system**: a valid-looking number that is not an order is stopped too.

> **Predict.** How many of the 31 numbers did Whisper base get right? And how many **wrong** numbers pass every check?

In [ ]:
from collections import Counter
results = []
for t in number_turns:
    found, problem = find_order_number(t["heard"], t["language"])
    if problem is None and found not in orders:
        problem = "not in the order system"
    right = found == t["number"]
    results.append({**t, "found": found, "problem": problem, "right": right})
n_right = sum(r["right"] for r in results)
wrong_passed = [r for r in results if not r["right"] and r["problem"] is None]
print(f"right: {n_right} of {len(results)}")
print("what stopped the others:", Counter(r["problem"] for r in results if not r["right"]))
print("wrong numbers that passed every check:", len(wrong_passed))

> **Check.** 12 of 31 right. Two of them are the wrong numbers that the "wrong number, then fixed" callers say on purpose: Whisper wrote them right, and the check digit stops them, as it should. The others were stopped: too short 14, no digits 4, check digit wrong 1. **0** wrong numbers passed every check: the IVR never acts on a misheard number. It asks again instead.

In [ ]:
for lang in ("en", "fr"):
    rs = [r for r in results if r["language"] == lang]
    print(f"{lang}: {sum(r['right'] for r in rs)} of {len(rs)} right")

> **Check.** English 10 of 13, French 2 of 18. Spoken French numbers are hard for Whisper base on these calls: the keypad is the real way in for French callers. (The lesson's chart comes from another run of Whisper base on the same turns, with 11 for English: Whisper on several CPU threads does not write exactly the same text on every run.)

## 4. Your turn: read it back in groups

> **Your turn.** Write `in_groups(digits)`: the digits in groups of three, with a space between digits and a comma and a space between groups. `"742158"` → `"7 4 2, 1 5 8"`; `"79147"` → `"7 9 1, 4 7"`.

In [ ]:
def in_groups(digits):
    """Read a number back in groups of three: '742158' -> '7 4 2, 1 5 8'."""
    return ", ".join(" ".join(digits[i:i + 3]) for i in range(0, len(digits), 3))

print(in_groups("742158"), "|", in_groups("79147"))

In [ ]:
expect_hash('in_groups on three numbers', [in_groups(x) for x in ("742158", "79147", "60313")], '291d26577d5440a8')

## 5. Your turn: yes or no, safely

> **Your turn.** Write `yes_no(text, language)`. Return `"no"` when the answer has a no-word (no, nope, wrong, not, incorrect, non, pas, faux), else `"yes"` when it has a yes-word (yes, yeah, yep, correct, right, oui, exact, exactement, ouais, or "c'est ça"), else `None`. Use `cp.normalise(text, "basic", language)` first. Check for a **no first**: "That's not right" has the yes-word "right".

In [ ]:
YES = {"yes", "yeah", "yep", "correct", "right", "oui", "exact", "exactement", "ouais"}
NO = {"no", "nope", "wrong", "not", "incorrect", "non", "pas", "faux"}

def yes_no(text, language):
    """'yes', 'no' or None. A no-word wins, so "that's not right" is a no."""
    words = set(cp.normalise(text, "basic", language).split())
    if words & NO:
        return "no"
    if words & YES or "c est ça" in cp.normalise(text, "basic", language):
        return "yes"
    return None

ANSWERS = [("Yes.", "en"), ("Yes, that's right.", "en"), ("No.", "en"), ("That's not right.", "en"),
           ("Not correct.", "en"), ("Correct.", "en"), ("Oui, c'est ça.", "fr"), ("Non, c'est faux.", "fr"),
           ("Exactement.", "fr"), ("Pardon?", "fr")]
print([yes_no(text, lang) for text, lang in ANSWERS])

In [ ]:
answers_out = [yes_no(text, lang) for text, lang in ANSWERS]

In [ ]:
expect_hash('yes_no on the ten answers', answers_out, 'd599b18b7488d4f4')

> **Check.** "That's not right." and "Not correct." are **no**; "Pardon?" is `None`, so the IVR asks again. The first version of the course's own rule had no "not", and said yes to "That's not right."

## 6. Failure case: six digits, and still wrong

In [ ]:
bad = [r for r in results if r["found"] and len(r["found"]) == 6 and not r["right"]]
for r in bad:
    print(r["call"], "said", r["number"], "| heard", repr(r["heard"]), "| found", r["found"], "->", r["problem"])

> **Check.** Wrong numbers with six digits: 1. The check digit stopped it. Without those two checks, the length alone would have let them through, and the IVR would have read out another caller's order.

## Next

You turned spoken numbers into digits, checked them three ways, and wrote a safe read-back and yes-or-no rule. The [lesson page](https://learning.nextia-ai.com/courses/speech/m07/numbers-confirmation-and-recovery/) has the recovery steps, the measurement of the number prompt, and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Test the IVR like a product](https://learning.nextia-ai.com/courses/speech/m07/test-the-ivr-like-a-product/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m07/numbers-confirmation-and-recovery/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.